# Mutual Fund Performance Analytics

## Objective

This notebook analyzes the performance of mutual fund schemes using
historical return metrics, risk-adjusted ratios, expense ratios,
benchmark performance, and fund rankings.

The analysis includes:

- Fund return comparison
- Risk-return analysis
- Alpha and Beta analysis
- Sharpe & Sortino Ratio
- Expense ratio impact
- Correlation analysis
- Benchmark comparison
- Fund scorecard

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path

In [2]:
BASE = Path("..")

In [3]:
performance = pd.read_csv(
    BASE / "data/processed/scheme_performance_clean.csv"
)

benchmark = pd.read_csv(
    BASE / "data/processed/10_benchmark_indices_clean.csv"
)

fund_master = pd.read_csv(
    BASE / "data/processed/01_fund_master_clean.csv"
)

## Data Inspection

In this section, we inspect the structure, shape, and missing values of the datasets before performing performance analysis.


In [4]:
# Display dataset shapes

print("Performance :", performance.shape)
print("Benchmark   :", benchmark.shape)
print("Fund Master :", fund_master.shape)

Performance : (40, 19)
Benchmark   : (8050, 3)
Fund Master : (40, 15)


In [5]:
performance.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [6]:
benchmark.head()


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [7]:
fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [8]:
performance.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     str    
 2   fund_house          40 non-null     str    
 3   category            40 non-null     str    
 4   plan                40 non-null     str    
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ratio_pct   4

In [9]:
performance.isnull().sum()

amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

In [10]:
# ============================================================
# Identify the Top 10 Mutual Funds based on 5-Year Returns
# ============================================================

# Sort all funds in descending order of 5-year return
# and select the top 10 performing schemes.

top10_return = (
    performance.sort_values(
        by="return_5yr_pct",
        ascending=False
    )
    .head(10)
)

In [11]:
# ============================================================
# Display the Top 10 Funds
# ============================================================

# Preview the top-performing mutual funds before plotting.

top10_return

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
29,101207,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Small Cap,Regular,24.93,22.38,23.80,20.54,1.84,0.97,0.90,1.47,25.0,-23.61,41613,1.53,5,Very High
27,119095,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Small Cap,Regular,21.97,20.98,22.62,20.47,0.51,1.00,0.84,1.40,25.0,-14.45,21545,1.38,4,Very High
17,118634,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Small Cap,Regular,21.30,20.15,21.88,19.35,0.80,1.03,0.81,1.14,25.0,-30.87,43630,1.53,4,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
39,149324,DSP Small Cap Fund - Regular - Growth,DSP Mutual Fund,Small Cap,Regular,20.20,20.08,20.61,19.39,0.69,0.98,0.80,1.23,25.0,-17.01,35124,1.52,4,Very High
38,149323,DSP Midcap Fund - Regular - Growth,DSP Mutual Fund,Mid Cap,Regular,14.12,17.16,19.00,16.14,1.02,0.98,0.90,1.50,19.0,-26.99,37835,1.61,4,High
26,119094,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,Mid Cap,Regular,14.88,15.18,18.94,13.76,1.42,1.00,0.80,1.18,19.0,-32.38,28996,1.38,5,High
21,120842,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,Mid Cap,Regular,17.12,18.23,17.75,16.32,1.91,1.00,0.96,1.27,19.0,-21.92,47469,1.56,4,High
7,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,HDFC Mutual Fund,Mid Cap,Regular,15.43,16.58,17.69,15.63,0.95,0.91,0.87,1.44,19.0,-13.67,23185,1.38,5,High


In [12]:
# ============================================================
# Create an Interactive Bar Chart
# ============================================================

# Plot the top 10 funds based on their 5-year returns.

fig = px.bar(
    top10_return,
    x="return_5yr_pct",
    y="scheme_name",
    color="fund_house",
    orientation="h",
    text="return_5yr_pct",
    title="Top 10 Mutual Funds by 5-Year Return"
)

# Improve chart appearance.

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    height=650
)

fig.show()

In [13]:
# ============================================================
# Save the Chart
# ============================================================

# Create the charts folder if it doesn't already exist.

CHARTS_DIR = BASE / "reports" / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Export the chart as a high-quality PNG image.

fig.write_image(
    CHARTS_DIR / "top10_5yr_return.png",
    scale=3
)

## 2. Sharpe Ratio Analysis

The Sharpe Ratio measures the risk-adjusted return of a mutual fund. A higher Sharpe Ratio indicates better returns for the level of risk taken.

In [14]:
# ============================================================
# Identify the Top 10 Funds based on Sharpe Ratio
# ============================================================

# Sort the funds by Sharpe Ratio in descending order
# and select the top 10 schemes.

top10_sharpe = (
    performance.sort_values(
        by="sharpe_ratio",
        ascending=False
    )
    .head(10)
)

In [15]:
# ============================================================
# Display the Top 10 Funds
# ============================================================

# Preview the funds with the highest Sharpe Ratios.

top10_sharpe

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
14,120507,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Liquid,Regular,8.89,7.68,7.94,5.83,1.85,0.26,7.68,10.37,0.5,-2.62,39116,0.74,5,Low
23,120844,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,Liquid,Regular,4.26,6.18,8.26,4.66,1.52,0.47,6.18,9.70,0.5,-3.81,27623,0.60,3,Low
30,101208,ABSL Liquid Fund - Regular - Growth,Aditya Birla Sun Life MF,Liquid,Regular,6.18,5.14,7.95,3.96,1.18,0.43,5.14,8.76,0.5,-3.66,38995,0.79,5,Low
9,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low
19,118636,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,Gilt,Regular,6.19,5.31,8.71,4.42,0.89,0.37,1.33,2.38,4.0,-2.23,30030,0.55,4,Low
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,Large Cap,Regular,15.12,14.81,12.68,13.19,1.62,0.96,1.06,1.66,14.0,-17.07,11361,1.46,5,Moderate
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,6434,1.55,5,Moderate
11,120504,ICICI Pru Bluechip Fund - Direct - Growth,ICICI Prudential MF,Large Cap,Direct,14.12,14.41,13.02,13.53,0.88,1.03,1.03,1.27,14.0,-26.59,41553,0.80,3,Moderate
15,118632,Nippon India Large Cap Fund - Regular - Growth,Nippon India MF,Large Cap,Regular,15.84,14.00,14.70,13.14,0.86,0.88,1.00,1.68,14.0,-16.07,20909,1.51,4,Moderate


In [16]:
# ============================================================
# Create an Interactive Bar Chart
# ============================================================

# Plot the top 10 funds based on Sharpe Ratio.

fig = px.bar(
    top10_sharpe,
    x="sharpe_ratio",
    y="scheme_name",
    color="fund_house",
    orientation="h",
    text="sharpe_ratio",
    title="Top 10 Mutual Funds by Sharpe Ratio"
)

# Improve chart appearance.

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    height=650
)

fig.show()

In [17]:
# ============================================================
# Save the Chart
# ============================================================

# Export the chart as a high-quality PNG image.

fig.write_image(
    CHARTS_DIR / "top10_sharpe_ratio.png",
    scale=3
)

### Observation

- Funds with higher Sharpe Ratios provide better risk-adjusted returns.
- The top-ranked funds have consistently generated strong returns while maintaining relatively lower volatility.

## 3. Sortino Ratio Analysis

The Sortino Ratio measures risk-adjusted returns by considering only downside volatility, making it a useful metric for evaluating downside risk.

In [18]:
# ============================================================
# Identify the Top 10 Funds based on Sortino Ratio
# ============================================================

# Sort the funds by Sortino Ratio in descending order
# and select the top 10 performing schemes.

top10_sortino = (
    performance.sort_values(
        by="sortino_ratio",
        ascending=False
    )
    .head(10)
)

top10_sortino

# ------------------------------------------------------------
# Observation:
# Funds with higher Sortino Ratios have delivered better
# downside risk-adjusted returns compared to other schemes.
# ------------------------------------------------------------

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
14,120507,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Liquid,Regular,8.89,7.68,7.94,5.83,1.85,0.26,7.68,10.37,0.5,-2.62,39116,0.74,5,Low
23,120844,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,Liquid,Regular,4.26,6.18,8.26,4.66,1.52,0.47,6.18,9.70,0.5,-3.81,27623,0.60,3,Low
30,101208,ABSL Liquid Fund - Regular - Growth,Aditya Birla Sun Life MF,Liquid,Regular,6.18,5.14,7.95,3.96,1.18,0.43,5.14,8.76,0.5,-3.66,38995,0.79,5,Low
9,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low
19,118636,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,Gilt,Regular,6.19,5.31,8.71,4.42,0.89,0.37,1.33,2.38,4.0,-2.23,30030,0.55,4,Low
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low
5,100016,HDFC Top 100 Fund - Regular Plan - Growth,HDFC Mutual Fund,Large Cap,Regular,10.94,14.84,11.32,14.06,0.78,0.97,1.06,1.70,14.0,-17.41,6434,1.55,5,Moderate
15,118632,Nippon India Large Cap Fund - Regular - Growth,Nippon India MF,Large Cap,Regular,15.84,14.00,14.70,13.14,0.86,0.88,1.00,1.68,14.0,-16.07,20909,1.51,4,Moderate
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,Large Cap,Regular,15.12,14.81,12.68,13.19,1.62,0.96,1.06,1.66,14.0,-17.07,11361,1.46,5,Moderate


In [19]:
# ============================================================
# Create an Interactive Bar Chart
# ============================================================

# Plot the Top 10 Funds based on Sortino Ratio.

fig = px.bar(
    top10_sortino,
    x="sortino_ratio",
    y="scheme_name",
    color="fund_house",
    orientation="h",
    text="sortino_ratio",
    title="Top 10 Mutual Funds by Sortino Ratio"
)

# Improve chart appearance.

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    height=650
)

fig.show()

# ------------------------------------------------------------
# Observation:
# The leading funds maintain strong returns while limiting
# downside volatility, resulting in higher Sortino Ratios.
# ------------------------------------------------------------

In [20]:
# ============================================================
# Save the Chart
# ============================================================

# Export the chart as a high-quality PNG image.

fig.write_image(
    CHARTS_DIR / "top10_sortino_ratio.png",
    scale=3
)

## 4. Alpha Analysis

Alpha measures the excess return generated by a mutual fund compared to its benchmark. A higher Alpha indicates that the fund has outperformed the benchmark after adjusting for market risk.

In [21]:
# ============================================================
# Identify the Top 10 Funds based on Alpha
# ============================================================

# Sort the funds by Alpha in descending order
# and select the top 10 performing schemes.

top10_alpha = (
    performance.sort_values(
        by="alpha",
        ascending=False
    )
    .head(10)
)

top10_alpha

# ------------------------------------------------------------
# Observation:
# Funds with higher Alpha have generated returns above their
# benchmark and indicate better fund management performance.
# ------------------------------------------------------------

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
9,100025,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Short Duration,Regular,6.83,7.37,6.41,5.39,1.98,0.44,1.84,2.79,4.0,-6.01,27953,0.56,3,Low
21,120842,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,Mid Cap,Regular,17.12,18.23,17.75,16.32,1.91,1.00,0.96,1.27,19.0,-21.92,47469,1.56,4,High
14,120507,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Liquid,Regular,8.89,7.68,7.94,5.83,1.85,0.26,7.68,10.37,0.5,-2.62,39116,0.74,5,Low
22,120843,Kotak Flexicap Fund - Regular - Growth,Kotak Mahindra MF,Flexi Cap,Regular,15.74,15.65,13.50,13.80,1.85,0.95,0.98,1.57,16.0,-19.50,35012,1.45,5,Moderately High
29,101207,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Small Cap,Regular,24.93,22.38,23.80,20.54,1.84,0.97,0.90,1.47,25.0,-23.61,41613,1.53,5,Very High
37,149322,DSP Top 100 Equity Fund - Regular - Growth,DSP Mutual Fund,Large Cap,Regular,11.96,12.82,12.35,11.00,1.82,0.91,0.92,1.63,14.0,-21.70,41828,1.54,5,Moderate
18,118635,Nippon India ETF Nifty 50 BeES,Nippon India MF,Index/ETF,Direct,10.14,11.77,12.31,9.97,1.80,1.04,0.91,1.24,13.0,-26.75,20284,0.89,5,Moderate
33,102887,UTI Flexi Cap Fund - Regular - Growth,UTI Mutual Fund,Flexi Cap,Regular,17.43,15.34,15.78,13.55,1.79,1.00,0.96,1.37,16.0,-12.14,17912,1.64,5,Moderately High
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
35,148568,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,Large & Mid Cap,Regular,14.91,14.56,15.68,12.86,1.70,0.99,0.91,1.55,16.0,-33.15,49046,1.52,5,Moderately High


In [22]:
# ============================================================
# Create an Interactive Bar Chart
# ============================================================

# Plot the Top 10 Funds based on Alpha.

fig = px.bar(
    top10_alpha,
    x="alpha",
    y="scheme_name",
    color="fund_house",
    orientation="h",
    text="alpha",
    title="Top 10 Mutual Funds by Alpha"
)

# Improve chart appearance.

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    height=650
)

fig.show()

# ------------------------------------------------------------
# Observation:
# Higher Alpha values indicate that these funds have
# consistently outperformed their benchmark.
# ------------------------------------------------------------

In [23]:
# ============================================================
# Save the Chart
# ============================================================

# Export the chart as a high-quality PNG image.

fig.write_image(
    CHARTS_DIR / "top10_alpha.png",
    scale=3
)

## 5. Beta Analysis

Beta measures a fund's sensitivity to market movements. A Beta close to 1 indicates that the fund moves similarly to the benchmark, while values above or below 1 indicate higher or lower volatility.

In [24]:
# ============================================================
# Identify the Funds with the Highest Beta
# ============================================================

# Sort the funds by Beta in descending order
# and select the top 10 schemes.

top10_beta = (
    performance.sort_values(
        by="beta",
        ascending=False
    )
    .head(10)
)

top10_beta

# ------------------------------------------------------------
# Observation:
# Funds with higher Beta are generally more sensitive to
# market movements and may experience larger fluctuations.
# ------------------------------------------------------------

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
8,125498,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Mid Cap,Direct,19.98,15.29,15.85,14.39,0.90,1.04,0.80,1.38,19.0,-32.22,18792,0.78,4,High
18,118635,Nippon India ETF Nifty 50 BeES,Nippon India MF,Index/ETF,Direct,10.14,11.77,12.31,9.97,1.80,1.04,0.91,1.24,13.0,-26.75,20284,0.89,5,Moderate
17,118634,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Small Cap,Regular,21.30,20.15,21.88,19.35,0.80,1.03,0.81,1.14,25.0,-30.87,43630,1.53,4,Very High
11,120504,ICICI Pru Bluechip Fund - Direct - Growth,ICICI Prudential MF,Large Cap,Direct,14.12,14.41,13.02,13.53,0.88,1.03,1.03,1.27,14.0,-26.59,41553,0.80,3,Moderate
28,101206,ABSL Frontline Equity Fund - Regular - Growth,Aditya Birla Sun Life MF,Large Cap,Regular,14.82,13.78,12.86,12.44,1.34,1.03,0.98,1.25,14.0,-15.07,23500,1.60,5,Moderate
16,118633,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,Large Cap,Direct,14.42,12.33,14.72,10.63,1.70,1.02,0.88,1.48,14.0,-18.14,39475,0.72,3,Moderate
12,120505,ICICI Pru Midcap Fund - Regular - Growth,ICICI Prudential MF,Mid Cap,Regular,14.02,18.08,17.55,17.19,0.89,1.00,0.95,1.45,19.0,-21.84,979,1.36,3,High
33,102887,UTI Flexi Cap Fund - Regular - Growth,UTI Mutual Fund,Flexi Cap,Regular,17.43,15.34,15.78,13.55,1.79,1.00,0.96,1.37,16.0,-12.14,17912,1.64,5,Moderately High
26,119094,Axis Midcap Fund - Regular - Growth,Axis Mutual Fund,Mid Cap,Regular,14.88,15.18,18.94,13.76,1.42,1.00,0.80,1.18,19.0,-32.38,28996,1.38,5,High


In [25]:
# ============================================================
# Create an Interactive Bar Chart
# ============================================================

# Plot the Top 10 Funds based on Beta.

fig = px.bar(
    top10_beta,
    x="beta",
    y="scheme_name",
    color="fund_house",
    orientation="h",
    text="beta",
    title="Top 10 Mutual Funds by Beta"
)

# Improve chart appearance.

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    height=650
)

fig.show()

# ------------------------------------------------------------
# Observation:
# Most equity funds have Beta values close to 1,
# indicating similar volatility to the market benchmark.
# ------------------------------------------------------------

In [26]:
# ============================================================
# Save the Chart
# ============================================================

# Export the chart as a high-quality PNG image.

fig.write_image(
    CHARTS_DIR / "top10_beta.png",
    scale=3
)

## 6. Maximum Drawdown Analysis

Maximum Drawdown represents the largest decline from a fund's peak value. Lower drawdowns indicate better downside protection during market corrections.

In [27]:
# ============================================================
# Maximum Drawdown Analysis
# ============================================================

# Sort the funds by Maximum Drawdown (higher values are better
# since drawdowns are stored as negative percentages).

top10_drawdown = (
    performance.sort_values(
        by="max_drawdown_pct",
        ascending=False
    )
    .head(10)
)

# Display the top funds.
top10_drawdown

# Create an interactive bar chart.

fig = px.bar(
    top10_drawdown,
    x="max_drawdown_pct",
    y="scheme_name",
    color="fund_house",
    orientation="h",
    text="max_drawdown_pct",
    title="Top Funds with Lowest Maximum Drawdown"
)

fig.update_layout(
    template="plotly_white",
    yaxis=dict(autorange="reversed"),
    height=650
)

fig.show()

# Save the chart.

fig.write_image(
    CHARTS_DIR / "maximum_drawdown.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# These funds experienced relatively smaller losses during
# market corrections and recovered more efficiently.
# ------------------------------------------------------------

## 7. Expense Ratio vs 5-Year Return

This analysis examines whether funds with higher expense ratios deliver better long-term returns.

In [28]:
# ============================================================
# Expense Ratio vs 5-Year Return Analysis
# ============================================================

# Create a scatter plot to compare expense ratio and
# 5-year returns across all mutual funds.

fig = px.scatter(
    performance,
    x="expense_ratio_pct",
    y="return_5yr_pct",
    color="category",
    size="aum_crore",
    hover_name="scheme_name",
    title="Expense Ratio vs 5-Year Return"
)

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()

# Save the chart.

fig.write_image(
    CHARTS_DIR / "expense_ratio_vs_return.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# No strong relationship is observed between expense ratio
# and long-term returns. Several low-cost funds have
# delivered competitive performance.
# ------------------------------------------------------------

## 8. Risk vs Return Analysis

This visualization compares annualized volatility with 5-year returns to understand the relationship between risk and performance.

In [29]:
# ============================================================
# Risk vs Return Analysis
# ============================================================

# Compare annualized volatility with 5-year returns.

fig = px.scatter(
    performance,
    x="std_dev_ann_pct",
    y="return_5yr_pct",
    color="category",
    size="aum_crore",
    hover_name="scheme_name",
    title="Risk vs Return Analysis"
)

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()

# Save the chart.

fig.write_image(
    CHARTS_DIR / "risk_vs_return.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# Equity funds generally offer higher returns at the cost of
# increased volatility, while debt funds remain relatively
# stable with lower returns.
# ------------------------------------------------------------

## 9. Alpha vs Beta Analysis

This chart compares Alpha and Beta values to understand fund performance relative to market risk.

In [30]:
# ============================================================
# Alpha vs Beta Analysis
# ============================================================

# Visualize Alpha and Beta for all mutual funds.

fig = px.scatter(
    performance,
    x="beta",
    y="alpha",
    color="category",
    size="aum_crore",
    hover_name="scheme_name",
    title="Alpha vs Beta Analysis"
)

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()

# Save the chart.

fig.write_image(
    CHARTS_DIR / "alpha_vs_beta.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# Most equity funds have Beta values close to 1. Funds with
# higher Alpha indicate better performance after adjusting
# for market risk.
# ------------------------------------------------------------

In [31]:
# ============================================================
# 7. Expense Ratio vs 5-Year Return Analysis
# ============================================================

# Create a scatter plot to compare expense ratio
# and 5-year returns across all mutual funds.

fig = px.scatter(
    performance,
    x="expense_ratio_pct",
    y="return_5yr_pct",
    color="category",
    size="aum_crore",
    hover_name="scheme_name",
    title="Expense Ratio vs 5-Year Return"
)

# Improve chart appearance.

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()

# Save the chart.

fig.write_image(
    CHARTS_DIR / "expense_ratio_vs_return.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# No strong relationship is observed between expense ratio
# and long-term returns. Several low-cost funds have
# delivered competitive performance.
# ------------------------------------------------------------

In [32]:
# ============================================================
# 8. Risk vs Return Analysis
# ============================================================

# Compare annualized volatility with 5-year returns.

fig = px.scatter(
    performance,
    x="std_dev_ann_pct",
    y="return_5yr_pct",
    color="category",
    size="aum_crore",
    hover_name="scheme_name",
    title="Risk vs Return Analysis"
)

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()

# Save the chart.

fig.write_image(
    CHARTS_DIR / "risk_vs_return.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# Equity funds generally provide higher returns with
# increased volatility, while debt funds remain more stable.
# ------------------------------------------------------------

In [33]:
# ============================================================
# 9. Alpha vs Beta Analysis
# ============================================================

# Visualize the relationship between Alpha and Beta.

fig = px.scatter(
    performance,
    x="beta",
    y="alpha",
    color="category",
    size="aum_crore",
    hover_name="scheme_name",
    title="Alpha vs Beta Analysis"
)

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()

# Save the chart.

fig.write_image(
    CHARTS_DIR / "alpha_vs_beta.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# Funds with higher Alpha have outperformed their benchmark,
# while Beta indicates their sensitivity to market movements.
# ------------------------------------------------------------

In [34]:
# ============================================================
# 10. Correlation Heatmap
# ============================================================

# Analyze the relationship between key performance metrics.

corr_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "expense_ratio_pct",
    "max_drawdown_pct"
]

corr_matrix = performance[corr_cols].corr()

fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Correlation Heatmap of Performance Metrics"
)

fig.update_layout(
    template="plotly_white",
    height=700
)

fig.show()

fig.write_image(
    CHARTS_DIR / "correlation_heatmap.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# Return metrics are positively correlated with each other,
# while higher volatility is generally associated with
# larger drawdowns.
# ------------------------------------------------------------

In [35]:
# ============================================================
# 11. Morningstar Rating Distribution
# ============================================================

# Count the number of funds in each Morningstar rating.

rating_count = (
    performance["morningstar_rating"]
    .value_counts()
    .sort_index()
    .reset_index()
)

rating_count.columns = ["Rating", "Funds"]

fig = px.bar(
    rating_count,
    x="Rating",
    y="Funds",
    text="Funds",
    title="Morningstar Rating Distribution"
)

fig.update_layout(
    template="plotly_white",
    height=550
)

fig.show()

fig.write_image(
    CHARTS_DIR / "morningstar_rating_distribution.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# Most mutual funds are concentrated between
# 3-star and 5-star ratings.
# ------------------------------------------------------------

In [36]:
# ============================================================
# 12. Benchmark Trend Analysis
# ============================================================

# Convert the benchmark date column to datetime.

benchmark["date"] = pd.to_datetime(benchmark["date"])

fig = px.line(
    benchmark,
    x="date",
    y="close_value",
    color="index_name",
    title="Benchmark Index Performance"
)

fig.update_layout(
    template="plotly_white",
    height=650
)

fig.show()

fig.write_image(
    CHARTS_DIR / "benchmark_trend.png",
    scale=3
)

# ------------------------------------------------------------
# Observation:
# The benchmark shows long-term growth despite
# periodic market corrections.
# ------------------------------------------------------------

In [37]:
# ============================================================
# 13. Fund Scorecard (0–100)
# ============================================================

# Rank funds across different performance metrics.

scorecard = performance.copy()

scorecard["return_rank"] = scorecard["return_3yr_pct"].rank(pct=True)
scorecard["sharpe_rank"] = scorecard["sharpe_ratio"].rank(pct=True)
scorecard["alpha_rank"] = scorecard["alpha"].rank(pct=True)
scorecard["expense_rank"] = 1 - scorecard["expense_ratio_pct"].rank(pct=True)
scorecard["drawdown_rank"] = scorecard["max_drawdown_pct"].rank(pct=True)

# Calculate the weighted score.

scorecard["fund_score"] = (
    0.30 * scorecard["return_rank"] +
    0.25 * scorecard["sharpe_rank"] +
    0.20 * scorecard["alpha_rank"] +
    0.15 * scorecard["expense_rank"] +
    0.10 * scorecard["drawdown_rank"]
) * 100

scorecard = scorecard.sort_values(
    "fund_score",
    ascending=False
)

scorecard.head(10)

fig = px.bar(
    scorecard.head(10),
    x="fund_score",
    y="scheme_name",
    color="fund_house",
    orientation="h",
    text="fund_score",
    title="Top 10 Mutual Funds by Composite Fund Score"
)

fig.update_layout(
    template="plotly_white",
    yaxis=dict(autorange="reversed"),
    height=650
)

fig.show()

fig.write_image(
    CHARTS_DIR / "fund_scorecard.png",
    scale=3
)

scorecard.to_csv(
    BASE / "reports" / "fund_scorecard.csv",
    index=False
)

# ------------------------------------------------------------
# Observation:
# The composite score rewards funds that combine
# strong returns, good risk-adjusted performance,
# lower expenses, and better downside protection.
# ------------------------------------------------------------

In [38]:
# ============================================================
# 14. Export Results and Display Final Summary
# ============================================================

# Export Alpha and Beta values.

performance[
    [
        "amfi_code",
        "scheme_name",
        "alpha",
        "beta"
    ]
].to_csv(
    BASE / "reports" / "alpha_beta.csv",
    index=False
)

# Display the Top 10 Funds based on the composite score.

print("=" * 70)
print("Top 10 Mutual Funds Based on Composite Fund Score")
print("=" * 70)

display(
    scorecard[
        [
            "scheme_name",
            "fund_house",
            "fund_score"
        ]
    ].head(10)
)

print("\nAll reports have been generated successfully.")
print(f"Charts saved to : {CHARTS_DIR}")
print(f"Fund Scorecard : {BASE/'reports'/'fund_scorecard.csv'}")
print(f"Alpha-Beta CSV : {BASE/'reports'/'alpha_beta.csv'}")

# ------------------------------------------------------------
# Final Summary
#
# • Analyzed mutual fund performance using return, risk,
#   Alpha, Beta, Sharpe Ratio, Sortino Ratio, expense ratio,
#   and maximum drawdown.
#
# • Compared funds using interactive visualizations and
#   generated a composite score for ranking.
#
# • Exported the required charts and CSV reports for further
#   analysis and reporting.
# ------------------------------------------------------------

Top 10 Mutual Funds Based on Composite Fund Score


,scheme_name,fund_house,fund_score
22,Kotak Flexicap Fund - Regular - Growth,Kotak Mahindra MF,71.3750
2,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,70.2500
14,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,70.1250
9,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,69.8750
21,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,67.8750
3,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,67.6250
34,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset MF,66.3125
29,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,64.6250
23,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,63.7500
33,UTI Flexi Cap Fund - Regular - Growth,UTI Mutual Fund,62.4375



All reports have been generated successfully.
Charts saved to : ..\reports\charts
Fund Scorecard : ..\reports\fund_scorecard.csv
Alpha-Beta CSV : ..\reports\alpha_beta.csv
